# Week 2, Day 1 — NumPy and pandas fundamentals

**Notebook:** `02_numpy_pandas_basics.ipynb`  
**Date:** 2026-09-16  
**Goal:** Rebuild the core data structures used in analytical Python and practice them on a small, controlled FX sample before touching the NBP API.

## Expected outcomes
1. I can create NumPy arrays, boolean masks, and vectorized calculations, and I know how they differ from Python lists.
2. I can build a time-indexed Series and a long-format DataFrame with explicit `datetime64`, `category`, and `float64` dtypes.
3. I can detect a missing rate and a duplicate currency–date key, then run a quality contract (`assert`) from a fresh kernel.

## Why this notebook exists
This is a rehearsal. Tomorrow the same ideas — types, long format, missing values, duplicates — will be applied to three years of EUR/PLN and USD/PLN rates.

In [1]:
import numpy as np
import pandas as pd
print(np.__version__)
print(pd.__version__)
rates_list = [4.31, 4.29, 4.32, 4.28, 4.30]
rates = np.array(rates_list, dtype = np.float64)
print(type(rates_list), type(rates))

print('Standard Python list:',rates_list)
print('Numpy ndarray:', rates)

# Compare shape
print(rates.ndim, rates.shape, rates.size, rates.dtype)



2.5.2
3.0.5
<class 'list'> <class 'numpy.ndarray'>
Standard Python list: [4.31, 4.29, 4.32, 4.28, 4.3]
Numpy ndarray: [4.31 4.29 4.32 4.28 4.3 ]
1 (5,) 5 float64


In [2]:
print('first:', rates[0])
print('last:', rates[-1])

# Middle of rates
middle_rates = rates[1:-1]

print('Slice:', middle_rates)
print('Shape of slice:', middle_rates.shape)


first: 4.31
last: 4.3
Slice: [4.29 4.32 4.28]
Shape of slice: (3,)


In [3]:
mask = rates > 4.30
print(mask)
print(mask.dtype)

high_rates = rates[mask]
print(high_rates)

[ True False  True False False]
bool
[4.31 4.32]


In [4]:
print('mean:', rates.mean())
print('min:', rates.min())
print('max:', rates.max())
print('std:', rates.std())

mean: 4.3
min: 4.28
max: 4.32
std: 0.0141421356237309


In [5]:
amounts_eur = np.array([100, 200, 500], dtype= np.float64)
amounts_pln = amounts_eur * 4.3
print(amounts_pln)

loop_result = [amount * 4.3 for amount in amounts_eur]
print(loop_result)

print(np.allclose(amounts_pln, loop_result))

[ 430.  860. 2150.]
[np.float64(430.0), np.float64(860.0), np.float64(2150.0)]
True


## Time index, Series, and long-format DataFrame

From here the unit of analysis is **one currency on one date**.  
That is the same grain we will use for three years of NBP rates.

In [6]:
dates = pd.to_datetime([
    "2026-08-31", "2026-09-01", "2026-09-02",
    "2026-09-03", "2026-09-04",
])
print(dates)
print(dates.dtype)

DatetimeIndex(['2026-08-31', '2026-09-01', '2026-09-02', '2026-09-03',
               '2026-09-04'],
              dtype='datetime64[us]', freq=None)
datetime64[us]


In [7]:
eur_series = pd.Series(rates, index = dates, name = 'EUR_PLN')
print(eur_series)
print(eur_series.index)
print(eur_series.dtype)

2026-08-31    4.31
2026-09-01    4.29
2026-09-02    4.32
2026-09-03    4.28
2026-09-04    4.30
Name: EUR_PLN, dtype: float64
DatetimeIndex(['2026-08-31', '2026-09-01', '2026-09-02', '2026-09-03',
               '2026-09-04'],
              dtype='datetime64[us]', freq=None)
float64


In [8]:
print(eur_series.loc["2026-09-01": "2026-09-03"])

2026-09-01    4.29
2026-09-02    4.32
2026-09-03    4.28
Name: EUR_PLN, dtype: float64


In [9]:
df = pd.DataFrame({
    "date": [
        "2026-08-31", "2026-08-31",
        "2026-09-01", "2026-09-01",
        "2026-09-02", "2026-09-02",
        "2026-09-03", "2026-09-03",
        "2026-09-04", "2026-09-04",
    ],
    "currency": ["EUR", "USD"] * 5,
    "rate": [4.31, 3.89, 4.29, 3.87, 4.32, 3.91, 4.28, 3.86, 4.30, 3.88],
})

print(df.shape)
print(df.head(10))
df.info()

(10, 3)
         date currency  rate
0  2026-08-31      EUR  4.31
1  2026-08-31      USD  3.89
2  2026-09-01      EUR  4.29
3  2026-09-01      USD  3.87
4  2026-09-02      EUR  4.32
5  2026-09-02      USD  3.91
6  2026-09-03      EUR  4.28
7  2026-09-03      USD  3.86
8  2026-09-04      EUR  4.30
9  2026-09-04      USD  3.88
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   date      10 non-null     str    
 1   currency  10 non-null     str    
 2   rate      10 non-null     float64
dtypes: float64(1), str(2)
memory usage: 372.0 bytes


In [10]:
print("PRZED")
print(df.dtypes)
print()

df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
df["currency"] = df["currency"].astype("category")
df["rate"] = pd.to_numeric(df["rate"], errors="raise").astype("float64")

print("PO")
print(df.dtypes)

PRZED
date            str
currency        str
rate        float64
dtype: object

PO
date        datetime64[us]
currency          category
rate               float64
dtype: object


In [11]:
df = df.sort_values(['currency', 'date']).reset_index(drop = True)
print(df)

df_indexed = df.set_index(['currency', 'date'])
print(df_indexed)

print(df_indexed.loc['EUR'])

        date currency  rate
0 2026-08-31      EUR  4.31
1 2026-09-01      EUR  4.29
2 2026-09-02      EUR  4.32
3 2026-09-03      EUR  4.28
4 2026-09-04      EUR  4.30
5 2026-08-31      USD  3.89
6 2026-09-01      USD  3.87
7 2026-09-02      USD  3.91
8 2026-09-03      USD  3.86
9 2026-09-04      USD  3.88
                     rate
currency date            
EUR      2026-08-31  4.31
         2026-09-01  4.29
         2026-09-02  4.32
         2026-09-03  4.28
         2026-09-04  4.30
USD      2026-08-31  3.89
         2026-09-01  3.87
         2026-09-02  3.91
         2026-09-03  3.86
         2026-09-04  3.88
            rate
date            
2026-08-31  4.31
2026-09-01  4.29
2026-09-02  4.32
2026-09-03  4.28
2026-09-04  4.30


In [12]:
df_problem = df.copy()
df_problem.loc[2, "rate"] = np.nan
df_problem = pd.concat([df_problem, df_problem.iloc[[0]]], ignore_index=True)

print(df_problem)
print("--- braki ---")
print(df_problem.isna().sum())
print("--- ile zduplikowanych kluczy ---")
print(df_problem.duplicated(["currency", "date"]).sum())
print("--- które wiersze są w duplikacie ---")
print(df_problem.loc[df_problem.duplicated(["currency", "date"], keep=False)])

         date currency  rate
0  2026-08-31      EUR  4.31
1  2026-09-01      EUR  4.29
2  2026-09-02      EUR   NaN
3  2026-09-03      EUR  4.28
4  2026-09-04      EUR  4.30
5  2026-08-31      USD  3.89
6  2026-09-01      USD  3.87
7  2026-09-02      USD  3.91
8  2026-09-03      USD  3.86
9  2026-09-04      USD  3.88
10 2026-08-31      EUR  4.31
--- braki ---
date        0
currency    0
rate        1
dtype: int64
--- ile zduplikowanych kluczy ---
1
--- które wiersze są w duplikacie ---
         date currency  rate
0  2026-08-31      EUR  4.31
10 2026-08-31      EUR  4.31


In [13]:
clean = (
    df_problem
    .dropna(subset=["rate"])
    .drop_duplicates(["currency", "date"], keep="first")
)
print(clean)

print(len(df), len(df_problem), len(clean))

        date currency  rate
0 2026-08-31      EUR  4.31
1 2026-09-01      EUR  4.29
3 2026-09-03      EUR  4.28
4 2026-09-04      EUR  4.30
5 2026-08-31      USD  3.89
6 2026-09-01      USD  3.87
7 2026-09-02      USD  3.91
8 2026-09-03      USD  3.86
9 2026-09-04      USD  3.88
10 11 9


In [14]:
assert df[["date", "currency", "rate"]].notna().all().all()
assert not df.duplicated(["currency", "date"]).any()
assert (df["rate"] > 0).all()
print("quality contract: OK")


quality contract: OK


In [15]:
summary = (df.groupby('currency', observed=True)['rate'].agg(['count', 'mean', 'min', 'max']))

summary

,count,mean,min,max
currency,,,,
EUR,5,4.300,4.28,4.32
USD,5,3.882,3.86,3.91
